[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/blhuillier/2026B_Galactic_Dynamics/blob/main/notebooks/chapter_04/nb_04_spherical_orbits.ipynb)

# Chapter 4 - Orbits in Spherical Potentials

**Galactic Dynamics · Sejong University · 2026B**

This notebook accompanies the numerical exercise of Chapter 4:

| Part | Exercise | Topic |
|------|----------|-------|
| A | 4.1 | Symplectic versus non-symplectic integration |

The kick-drift-kick leapfrog is derived in Section 4.8.2.1 of the notes, as
the composition of two exactly solvable Hamiltonian flows. Here it is tested
on the scaled pendulum

$$ H(\theta, p) = \tfrac12 p^2 - \cos\theta , $$

the system of Figure 3.2, with the separatrix at $H = 1$.

Cells marked **TODO** are yours to complete. Everything else is given.


> **Running on Google Colab?** This notebook uses the `galdyn` toolkit from the
> course repository. Run the cell below first - it clones the repo and adds it
> to your Python path. On a local install (via `uv`), this cell does nothing.

In [ ]:
import os
import sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.exists("2026B_Galactic_Dynamics"):
        !git clone -q https://github.com/blhuillier/2026B_Galactic_Dynamics.git
    sys.path.insert(0, "2026B_Galactic_Dynamics/src")
    %pip install -q astropy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from galdyn.plotting import set_latex_style

set_latex_style(use_tex=False)

rng = np.random.default_rng(20268)

---
## Part A - Exercise 4.1: symplectic versus non-symplectic integration

The pendulum in first-order form is

$$ \dot\theta = p, \qquad \dot p = -\sin\theta . $$

Both integrators below take the same right-hand side and the same step, so any
difference between them is structural, not a matter of accuracy.

In [ ]:
def pendulum_rhs(state):
    """Return d(theta, p)/dt for H = p^2/2 - cos(theta)."""
    theta, p = state
    return np.array([p, -np.sin(theta)])


def pendulum_energy(theta, p):
    """Return H = p^2/2 - cos(theta)."""
    return 0.5 * p**2 - np.cos(theta)


def rk4(state0, dt, n_steps):
    """Classical fourth-order Runge-Kutta. Not symplectic."""
    out = np.empty((n_steps + 1, 2))
    out[0] = state0
    for n in range(n_steps):
        y = out[n]
        k1 = pendulum_rhs(y)
        k2 = pendulum_rhs(y + 0.5 * dt * k1)
        k3 = pendulum_rhs(y + 0.5 * dt * k2)
        k4 = pendulum_rhs(y + dt * k3)
        out[n + 1] = y + dt * (k1 + 2 * k2 + 2 * k3 + k4) / 6.0
    return out


def leapfrog(state0, dt, n_steps):
    """Second-order kick-drift-kick leapfrog (Section 4.8.2.1). Symplectic."""
    out = np.empty((n_steps + 1, 2))
    out[0] = state0
    theta, p = state0
    for n in range(n_steps):
        p_half = p - 0.5 * dt * np.sin(theta)
        theta = theta + dt * p_half
        p = p_half - 0.5 * dt * np.sin(theta)
        out[n + 1] = (theta, p)
    return out

In [ ]:
state0 = np.array([1.0, 0.0])
dt = 0.05
n_periods = 10_000
n_steps = int(n_periods * 2 * np.pi / dt)

traj_rk4 = rk4(state0, dt, n_steps)
traj_lf = leapfrog(state0, dt, n_steps)
t = dt * np.arange(n_steps + 1)

print(f"{n_steps:,} steps, t_max = {t[-1]:.3g}")

**TODO A.1** - Plot $|H(t) - H(0)|$ against $t$ on logarithmic axes for both
integrators. Which is more accurate over one period? Over $10^4$? Describe the
*shape* of each curve, not just its size.

In [ ]:
# TODO: compute the energy error for each trajectory and plot it.
#
#   err_rk4 = np.abs(pendulum_energy(*traj_rk4.T) - pendulum_energy(*state0))
#   err_lf  = ...
#
# Use ax.loglog, and subsample (e.g. [::200]) so the figure stays light.

**TODO A.2** - Overlay the two trajectories in the $(\theta, p)$ plane. One
stays on a closed curve, the other does not. Connect what you see to
Exercise 3.7 on phase-space volume.

In [ ]:
# TODO: phase-plane overlay. Plot the first ~20 periods and the last ~20
# periods of each trajectory in different colours to make the drift visible.

**TODO A.3** - Halve `dt` and confirm the order of each method: the leapfrog
energy *envelope* should shrink by 4, the Runge-Kutta per-step error by 16.
Does the long-term ranking of the two methods change?

In [ ]:
# TODO: loop over dt in [0.1, 0.05, 0.025, 0.0125], measure
# max|H - H0| over a fixed physical time, and fit the slope on log-log axes.

**TODO A.4** - Repeat with an initial condition just inside the separatrix,
$H(0) = 0.999$. Something goes wrong. Explain why it is not the integrator's
fault. *Hint: Exercise 3.8, part 2.*

In [ ]:
# TODO: choose theta0 = 0 and solve p0 from H0 = 0.999, then integrate.
# Watch whether the orbit stays a libration or escapes into circulation.

**TODO A.5** - The leapfrog does not conserve $H$, but it conserves almost
exactly the shadow Hamiltonian of Section 4.8.2.1. For the pendulum,
$\Phi = -\cos\theta$, it reads

$$ \tilde H = H + \Delta t^2 \left( \tfrac{1}{12} p^2 \cos\theta
- \tfrac{1}{24} \sin^2\theta \right) + \mathcal{O}(\Delta t^4) . $$

Evaluate $\tilde H$ along the leapfrog trajectory and compare its variation
with that of $H$. How does the ratio change when you halve `dt`? Why does the
same test fail for the Runge-Kutta trajectory?


In [ ]:
def shadow_energy(theta, p, dt):
    """Shadow Hamiltonian of the kick-drift-kick leapfrog, to O(dt^2)."""
    raise NotImplementedError("TODO A.5")


# TODO: compare np.ptp(pendulum_energy(*traj_lf.T)) with
# np.ptp(shadow_energy(*traj_lf.T, dt)), and repeat for dt / 2.
